# Semi-supervised NN rescoring of a MuMDIA PIN

Ingests a Percolator `.pin` (MuMDIA `features`/`compete` output) and rescores it with a
**PyTorch MLP** trained in the **Percolator/mokapot semi-supervised** scheme, i.e. the
DIA-NN-style nonlinear classifier over the same feature set.

Algorithm (per cross-validation fold, so every PSM is scored by a model that never saw it):
1. Initialise a score from the single best feature+sign (most targets at `train_fdr`).
2. Repeat `n_iter` times: recompute target-decoy q-values on the training folds, take
   **target PSMs with q <= train_fdr as positives** and **all decoys as negatives**, train
   the MLP from scratch on that labelled set, rescore the training folds.
3. Score the held-out fold with the final model.

Final target-decoy q-values are computed at PSM and peptide level; the peptide count at
1% FDR is the number to compare against mokapot (linear) on the same PIN.

**Note.** The NN introduces run-to-run nondeterminism even with seeds set (CPU thread
scheduling, cuDNN off here). Treat the peptide count as approximate (+/- a few tens).

In [1]:
# ---- parameters ----
# PIN_PATH = r'C:/proteobench/out_ecoli_ox/run.pin'    # apex-gated pool (107k): NN=10,195, mokapot=9,928
PIN_PATH   = r'C:/proteobench/out_ox_gateoff/x.pin'   # GATE-OFF pool (503k, no spectral gate)
N_FOLDS    = 3        # cross-validation folds (mokapot default)
N_ITER     = 5        # semi-supervised self-training iterations
TRAIN_FDR  = 0.01     # positive-selection FDR during training
EPOCHS     = 25       # NN epochs per iteration
HIDDEN     = [128, 64]
DROPOUT    = 0.3
LR         = 1e-3
WEIGHT_DECAY = 1e-4
BATCH      = 4096
SEED       = 0

# optional DIA-NN concordance (set to None to skip)
DIANN_REPORT = r'C:/Users/robbi/OneDrive - UGent/MuMDIA_NG/out_diann/report.tsv'
DIANN_TAG    = 'ECOLI'   # substring in Proteins marking the target proteome
MOKAPOT_BASELINE = 9928  # apex-gated mokapot @1% (the number to beat; gate-off has no mokapot ref)

In [2]:
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(False)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)

device: cuda | torch 2.5.1+cu121


## Parse the PIN

Columns are `SpecId, Label, ScanNr, ExpMass, CalcMass, <features...>, Peptide, Proteins`.
`ExpMass`/`CalcMass` are Percolator bookkeeping, not features. Folds are assigned by a hash
of the **stripped peptide** so all PSMs of a peptide (and its paired decoy) stay together,
avoiding train/test leakage.

In [3]:
pin = pd.read_csv(PIN_PATH, sep='\t')
NON_FEATURE = {'SpecId', 'Label', 'ScanNr', 'ExpMass', 'CalcMass', 'Peptide', 'Proteins'}
feat_cols = [c for c in pin.columns if c not in NON_FEATURE]
print(f'{len(pin):,} PSMs | {len(feat_cols)} features')

y = (pin['Label'].to_numpy() == 1).astype(np.float32)          # 1 target, 0 decoy
X = np.nan_to_num(pin[feat_cols].to_numpy(np.float32), nan=0.0, posinf=0.0, neginf=0.0)

STRIP = lambda p: re.sub(r'\[[^\]]*\]', '', str(p)).strip('-.').split('.')[-1] if '.' in str(p) else str(p)
def strip_pep(p):
    s = re.sub(r'\[[^\]]*\]', '', str(p))          # drop mods
    s = re.sub(r'^[A-Z-]\.', '', s); s = re.sub(r'\.[A-Z-]$', '', s)  # drop flanks
    return s
pin['strip'] = pin['Peptide'].map(strip_pep)
pin['pep_mod'] = pin['Peptide'].map(lambda p: re.sub(r'^[A-Z-]\.|\.[A-Z-]$', '', str(p)))

# deterministic fold by peptide hash (keeps a peptide's PSMs in one fold)
import hashlib
def h(s):
    return int(hashlib.md5(s.encode()).hexdigest(), 16)
fold = pin['strip'].map(lambda s: h(s) % N_FOLDS).to_numpy()
print('fold sizes:', np.bincount(fold))

# robust standardisation (median / IQR, clip) for stable NN training
med = np.median(X, axis=0)
iqr = np.subtract(*np.percentile(X, [75, 25], axis=0)); iqr[iqr == 0] = 1.0
Xs = np.clip((X - med) / iqr, -8, 8).astype(np.float32)

503,251 PSMs | 379 features


C:\Users\robbi\AppData\Local\Temp\ipykernel_44372\3106957684.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pin['strip'] = pin['Peptide'].map(strip_pep)


C:\Users\robbi\AppData\Local\Temp\ipykernel_44372\3106957684.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pin['pep_mod'] = pin['Peptide'].map(lambda p: re.sub(r'^[A-Z-]\.|\.[A-Z-]$', '', str(p)))


fold sizes: [167533 167862 167856]


In [4]:
def tda_q(scores, is_target):
    """Target-decoy q-values. scores desc; FDR=(decoys+1)/max(1,targets), q=running min."""
    order = np.argsort(-scores, kind='stable')
    t = is_target[order].astype(float)
    ct = np.cumsum(t); cd = np.cumsum(1 - t)
    fdr = (cd + 1) / np.maximum(ct, 1)
    q = np.minimum.accumulate(fdr[::-1])[::-1]
    out = np.empty_like(q); out[order] = q
    return out

def n_targets_at(scores, is_target, fdr=0.01):
    q = tda_q(scores, is_target)
    return int(((q <= fdr) & (is_target == 1)).sum())

def peptide_count(scores, is_target, pep, fdr=0.01):
    """Best PSM per peptide, then target-decoy q at peptide level."""
    df = pd.DataFrame({'s': scores, 't': is_target, 'p': pep})
    best = df.sort_values('s', ascending=False).drop_duplicates('p')
    q = tda_q(best['s'].to_numpy(), best['t'].to_numpy())
    keep = (q <= fdr) & (best['t'].to_numpy() == 1)
    return int(keep.sum()), best.loc[keep, 'p']

In [5]:
# initial direction: best single feature+sign by targets at TRAIN_FDR (mokapot-style)
best_feat, best_sign, best_n = None, 1, -1
for j in range(Xs.shape[1]):
    for sign in (1, -1):
        n = n_targets_at(sign * Xs[:, j], y, TRAIN_FDR)
        if n > best_n:
            best_n, best_feat, best_sign = n, j, sign
print(f'init feature: {feat_cols[best_feat]} sign {best_sign:+d} -> {best_n} targets @ {TRAIN_FDR:.0%}')
init_score = best_sign * Xs[:, best_feat]

init feature: spectral_entropy_similarity_topk sign +1 -> 8571 targets @ 1%


In [6]:
class MLP(nn.Module):
    def __init__(self, d_in, hidden, p):
        super().__init__()
        layers, d = [], d_in
        for hdim in hidden:
            layers += [nn.Linear(d, hdim), nn.BatchNorm1d(hdim), nn.ReLU(), nn.Dropout(p)]
            d = hdim
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_model(Xtr, ytr, epochs, pos_weight):
    torch.manual_seed(SEED)
    m = MLP(Xtr.shape[1], HIDDEN, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    lossf = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=DEVICE))
    Xt = torch.from_numpy(Xtr).to(DEVICE); yt = torch.from_numpy(ytr).to(DEVICE)
    n = len(Xt)
    for _ in range(epochs):
        m.train(); perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad(); loss = lossf(m(Xt[idx]), yt[idx]); loss.backward(); opt.step()
    return m

@torch.no_grad()
def score_model(m, Xarr):
    m.eval()
    return m(torch.from_numpy(Xarr).to(DEVICE)).cpu().numpy()

In [7]:
# semi-supervised cross-validated training
final = np.zeros(len(Xs), np.float32)
for f in range(N_FOLDS):
    tr = fold != f; te = fold == f
    Xtr, ytr = Xs[tr], y[tr]
    score_tr = init_score[tr].copy()
    model = None
    for it in range(N_ITER):
        q = tda_q(score_tr, ytr)
        pos = (q <= TRAIN_FDR) & (ytr == 1)
        neg = ytr == 0
        sel = pos | neg
        pw = float(neg.sum()) / max(1.0, float(pos.sum()))     # balance rare positives
        model = train_model(Xtr[sel], ytr[sel], EPOCHS, pw)
        score_tr = score_model(model, Xtr)
        print(f'fold {f} iter {it}: {int(pos.sum())} positives, {int(neg.sum())} decoys, '
              f'train targets@1% = {n_targets_at(score_tr, ytr, TRAIN_FDR)}')
    final[te] = score_model(model, Xs[te])
print('CV scoring done')

fold 0 iter 0: 5701 positives, 163106 decoys, train targets@1% = 7431


fold 0 iter 1: 7431 positives, 163106 decoys, train targets@1% = 7732


fold 0 iter 2: 7732 positives, 163106 decoys, train targets@1% = 8022


fold 0 iter 3: 8022 positives, 163106 decoys, train targets@1% = 8138


fold 0 iter 4: 8138 positives, 163106 decoys, train targets@1% = 8176


...

fold 2 iter 1: 7168 positives, 162700 decoys, train targets@1% = 7664


fold 2 iter 2: 7664 positives, 162700 decoys, train targets@1% = 8034


fold 2 iter 3: 8034 positives, 162700 decoys, train targets@1% = 8125


fold 2 iter 4: 8125 positives, 162700 decoys, train targets@1% = 8143


CV scoring done


In [8]:
# final target-decoy q-values
psm_n = n_targets_at(final, y, 0.01)
pep_n, pep_ids = peptide_count(final, y, pin['pep_mod'].to_numpy(), 0.01)
print(f'PSMs   @1% FDR: {psm_n}')
print(f'peptides @1% FDR (NN): {pep_n}   | mokapot baseline: {MOKAPOT_BASELINE}   | delta {pep_n - MOKAPOT_BASELINE:+d}')

PSMs   @1% FDR: 11819
peptides @1% FDR (NN): 10447   | mokapot baseline: 9928   | delta +519


In [9]:
# optional: DIA-NN E.coli concordance
if DIANN_REPORT:
    dn = pd.read_csv(DIANN_REPORT, sep='\t', usecols=['Stripped.Sequence', 'Protein.Names', 'Q.Value'])
    dn = dn[(dn['Q.Value'] <= 0.01) & (dn['Protein.Names'].astype(str).str.contains(DIANN_TAG, na=False))]
    diann = set(dn['Stripped.Sequence'])
    # NN target peptides that are E.coli (Proteins carries the tag)
    q = tda_q(final, y)
    df = pin.assign(q=q, s=final)
    tgt = df[(df.Label == 1) & (df.q <= 0.01) & df.Proteins.astype(str).str.contains(DIANN_TAG, na=False)]
    nn_strip = set(tgt['strip'])
    rec = nn_strip & diann
    print(f'DIA-NN E.coli @1%: {len(diann)}')
    print(f'NN E.coli peptides: {len(nn_strip)} | recovered {len(rec)} = {100*len(rec)/len(diann):.1f}% of DIA-NN '
          f'| concordance {100*len(rec)/max(1,len(nn_strip)):.1f}%')

DIA-NN E.coli @1%: 11642
NN E.coli peptides: 10327 | recovered 9707 = 83.4% of DIA-NN | concordance 94.0%


C:\Users\robbi\AppData\Local\Temp\ipykernel_44372\2592420269.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pin.assign(q=q, s=final)
C:\Users\robbi\AppData\Local\Temp\ipykernel_44372\2592420269.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = pin.assign(q=q, s=final)


## Notes

- **Fair comparison:** run against the same PIN mokapot scored (`out_ecoli_ox/run.pin`, apex-gated pool,
  mokapot = 9,928 peptides). Any lift is the nonlinear classifier extracting more from the 379 features.
- **q-value estimator:** `(decoys+1)/max(1,targets)` with running-min, standard TDA. mokapot's estimator
  differs slightly, so treat the absolute count as comparable-not-identical; the delta is the signal.
- **Leakage:** folds are split by stripped peptide, so a target and its paired decoy never straddle
  train/test. Standardisation is global (negligible leakage).
- **Levers to try:** deeper/wider `HIDDEN`, more `N_ITER`, ensembling several seeds and averaging scores
  (reduces NN variance, closer to DIA-NN's NN ensemble), focal/ranking loss instead of BCE, or feeding a
  gate-off PIN (larger pool) to test whether the NN tolerates the FDR-flood better than the linear model.